# Experiment 8: Text Application using Transformer

**Course:** AML ZC417 – Introduction to Deep Learning
**Module Reference:** Module 9 — Transformers
**Duration:** 2 hours

---

## Aim
To build a Transformer encoder from its core components (self-attention, multi-head attention, positional encoding) for text classification, and to compare its performance against the RNN-based models from Experiment 7 on the same sentiment classification task.

## Learning Outcomes
By the end of this experiment, you will be able to:
1. Explain the self-attention mechanism and why it allows a model to relate any two positions in a sequence directly.
2. Explain why positional encoding is necessary in a Transformer, unlike in an RNN.
3. Build a Transformer encoder block (multi-head attention + feed-forward network) using Keras.
4. Train a Transformer-based text classifier and compare it against SimpleRNN/LSTM/Bidirectional LSTM (Experiment 7).
5. Visualize attention weights to interpret which words a Transformer attends to.
6. Explain the practical trade-offs between RNN-based and Transformer-based architectures.


## Part 0 — Setup and Dataset

We use the same **IMDB Movie Reviews dataset** as Experiment 7, so that results are directly comparable to the SimpleRNN, LSTM, and Bidirectional LSTM models built there.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("TensorFlow version:", tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)

VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 32


In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

x_train_full_padded = keras.preprocessing.sequence.pad_sequences(x_train_full, maxlen=MAX_LEN)
x_test_padded = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=MAX_LEN)

x_train, x_val = x_train_full_padded[:20000], x_train_full_padded[20000:]
y_train, y_val = y_train_full[:20000], y_train_full[20000:]

print("Train shape:", x_train.shape, "| Val shape:", x_val.shape, "| Test shape:", x_test_padded.shape)

word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0], reverse_word_index[1], reverse_word_index[2], reverse_word_index[3] = "<PAD>", "<START>", "<UNK>", "<UNUSED>"

def decode_review(encoded_review):
    return " ".join(reverse_word_index.get(i, "?") for i in encoded_review)


---
## Part 1 — Why Move Beyond RNNs?

RNNs (Experiments 6 and 7) process a sequence **one timestep at a time**, which creates two practical problems:

1. **Sequential computation** -- each timestep depends on the previous one's output, so RNN training cannot be parallelized across the time dimension, making it slow on long sequences.
2. **Long-range dependencies are still hard** -- even LSTMs/GRUs, despite their gating mechanisms, can struggle to directly relate two words that are very far apart in a sequence, since information must still pass through every intermediate timestep.

A **Transformer** solves both problems with **self-attention**: every position in the sequence can directly attend to (look at) every other position in a single step, regardless of distance, and this computation can be fully parallelized across positions during training.


---
## Part 2 — Understanding Self-Attention

Self-attention computes, for each word, a weighted combination of **all** words in the sequence (including itself), where the weights indicate how relevant each other word is to understanding the current one. This is computed using three learned projections of the input: **Query (Q)**, **Key (K)**, and **Value (V)**.

The attention score between position *i* and position *j* is computed as the (scaled) dot product of Query_i and Key_j; these scores are passed through a softmax to get attention weights, which are then used to combine the Value vectors.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Let's compute this manually on a tiny toy example to make the mechanism concrete before using Keras's built-in layer.


In [ ]:
# Toy example: 4 "words", each represented by a 3-dimensional vector (normally this would be an embedding)
np.random.seed(0)
seq_len, d_model = 4, 3
X_toy = np.random.randn(seq_len, d_model)

# For this toy example, use random Q, K, V projection matrices
Wq = np.random.randn(d_model, d_model) * 0.5
Wk = np.random.randn(d_model, d_model) * 0.5
Wv = np.random.randn(d_model, d_model) * 0.5

Q = X_toy @ Wq
K = X_toy @ Wk
V = X_toy @ Wv

# Scaled dot-product attention
scores = (Q @ K.T) / np.sqrt(d_model)

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

attention_weights = softmax(scores, axis=-1)
output = attention_weights @ V

print("Attention weights (each row sums to 1):\n", np.round(attention_weights, 3))
print("\nOutput shape:", output.shape, "(same shape as input -- one output vector per input position)")


In [ ]:
# Visualize the toy attention weights as a heatmap
plt.figure(figsize=(5, 4))
plt.imshow(attention_weights, cmap="viridis")
plt.colorbar(label="Attention weight")
plt.xlabel("Key position (attended TO)")
plt.ylabel("Query position (attending FROM)")
plt.title("Toy Self-Attention Weights")
plt.xticks(range(seq_len)); plt.yticks(range(seq_len))
plt.show()


**What to look for:** Each row sums to 1 (it is a probability distribution over which positions to attend to). Row *i* shows how much position *i* "looks at" every other position (including itself) when forming its output representation. Unlike an RNN, position 0 can directly attend to position 3 with no intermediate steps.


---
## Part 3 — Positional Encoding

Self-attention treats the input as an **unordered set** of vectors -- nothing in the attention formula depends on word order. Without extra information, "the dog bit the man" and "the man bit the dog" would produce identical attention computations! Transformers fix this by adding a **positional encoding** vector to each word's embedding, injecting information about its position in the sequence.

A common scheme uses sine and cosine functions of different frequencies:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$


In [ ]:
def get_positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return angle_rads

pe = get_positional_encoding(MAX_LEN, EMBEDDING_DIM)

plt.figure(figsize=(9, 4))
plt.imshow(pe.T, cmap="RdBu", aspect="auto")
plt.xlabel("Position in sequence"); plt.ylabel("Embedding dimension")
plt.title("Positional Encoding Pattern")
plt.colorbar(label="Value")
plt.show()


**What to look for:** Each row is a different sine/cosine wave frequency, and each column (position) gets a unique combination of values across all dimensions -- giving the model a way to distinguish position 5 from position 50 from position 150.


---
## Part 4 — Building a Transformer Encoder Block

A standard Transformer encoder block consists of: **Multi-Head Self-Attention -> Add & Normalize -> Feed-Forward Network -> Add & Normalize**. "Multi-head" means the attention computation (Part 2) is run several times in parallel with different learned projections, allowing the model to attend to different types of relationships simultaneously; the results are then concatenated.

We use Keras's built-in `MultiHeadAttention` layer, which implements exactly the scaled dot-product attention mechanism from Part 2, scaled up with multiple heads.


In [ ]:
class TransformerEncoderBlock(keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            keras.layers.Dense(ff_dim, activation="relu"),
            keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = keras.layers.Dropout(dropout_rate)
        self.dropout2 = keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_output = self.attention(query=inputs, value=inputs, key=inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)          # residual connection + normalize

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)              # residual connection + normalize

print("TransformerEncoderBlock defined.")


**Note the two residual ("Add") connections**: the input is added back to the attention output, and again to the feed-forward output, before normalizing. These residual connections (the same idea used in ResNet-style CNNs) help gradients flow through many stacked layers and are essential for training deep Transformers.


---
## Part 5 — Assembling the Full Transformer Classifier


In [ ]:
class PositionalEmbedding(keras.layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = keras.layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

def build_transformer_classifier(embed_dim=32, num_heads=2, ff_dim=32):
    inputs = keras.layers.Input(shape=(MAX_LEN,))
    x = PositionalEmbedding(MAX_LEN, VOCAB_SIZE, embed_dim)(inputs)
    x = TransformerEncoderBlock(embed_dim, num_heads, ff_dim)(x)
    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

transformer_model = build_transformer_classifier()
transformer_model.summary()


**Reading the architecture:** `PositionalEmbedding` combines a token embedding (as in Experiment 7) with a *learned* positional embedding (an alternative to the fixed sine/cosine scheme in Part 3 -- both are valid, and learned positional embeddings are common in practice). `GlobalAveragePooling1D` collapses the sequence of per-position vectors into a single fixed-length vector by averaging, similar in spirit to how the RNN's final hidden state summarized the whole sequence in Experiment 7.


In [ ]:
history_transformer = transformer_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=8,
    batch_size=64,
    verbose=2,
)


In [ ]:
transformer_test_loss, transformer_test_acc = transformer_model.evaluate(x_test_padded, y_test, verbose=0)
print(f"Transformer -- Test accuracy: {transformer_test_acc:.4f}, Test loss: {transformer_test_loss:.4f}")


---
## Part 6 — Comparing Against Experiment 7's RNN Models

Enter the test accuracy values you recorded for SimpleRNN, LSTM, and Bidirectional LSTM from Experiment 7 below, to compare all four architectures on the identical task, data split, and vocabulary/sequence-length settings.


In [ ]:
# TODO: replace these placeholder values with YOUR results from Experiment 7
rnn_test_acc = 0.75          # <-- replace with your Experiment 7 SimpleRNN test accuracy
lstm_test_acc = 0.85         # <-- replace with your Experiment 7 LSTM test accuracy
bilstm_test_acc = 0.86       # <-- replace with your Experiment 7 Bidirectional LSTM test accuracy

models = ["SimpleRNN", "LSTM", "BiLSTM", "Transformer"]
accs = [rnn_test_acc, lstm_test_acc, bilstm_test_acc, transformer_test_acc]

plt.figure(figsize=(7, 4))
bars = plt.bar(models, accs, color=["gray", "steelblue", "darkorange", "seagreen"])
plt.ylabel("Test Accuracy")
plt.title("RNN Variants vs. Transformer -- Sentiment Classification")
plt.ylim(0, 1)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f"{acc:.3f}", ha="center")
plt.tight_layout()
plt.show()


**What to look for:** On a moderate-sized dataset like this (25,000 training reviews) with only a few epochs, a small Transformer often performs comparably to LSTM/BiLSTM rather than dramatically better -- the Transformer's real advantages (training speed via parallelization, and superior performance at much larger scale/data) are less visible in a small 2-hour lab setting than in production-scale NLP systems, where Transformers are now dominant.


---
## Part 7 — Visualizing What the Transformer Attends To

We can extract the attention weights from the trained model's `MultiHeadAttention` layer for a specific review, to see which words the model focused on when forming its representation.


In [ ]:
# Rebuild a small model that exposes attention scores for inspection
sample_idx = 0
sample_review = x_test_padded[sample_idx : sample_idx + 1]

# Access the trained encoder block's attention layer directly
encoder_block = transformer_model.layers[2]   # PositionalEmbedding -> TransformerEncoderBlock -> ...
pos_embed_layer = transformer_model.layers[1]

embedded_input = pos_embed_layer(sample_review)
_, attention_scores = encoder_block.attention(
    query=embedded_input, value=embedded_input, key=embedded_input, return_attention_scores=True
)
print("Attention scores shape (batch, num_heads, query_pos, key_pos):", attention_scores.shape)


In [ ]:
# Average across heads, then plot a heatmap for the first 30 tokens (most reviews are padded at the start)
avg_attention = tf.reduce_mean(attention_scores[0], axis=0).numpy()   # (seq_len, seq_len)

original_review = x_test[sample_idx]
padded_review = x_test_padded[sample_idx]
start = MAX_LEN - len(original_review) if len(original_review) < MAX_LEN else 0
words = [reverse_word_index.get(idx, "?") for idx in padded_review[start:start+30]]

plt.figure(figsize=(10, 8))
plt.imshow(avg_attention[start:start+30, start:start+30], cmap="viridis")
plt.xticks(range(len(words)), words, rotation=90, fontsize=8)
plt.yticks(range(len(words)), words, fontsize=8)
plt.title("Average Attention Weights (first 30 real tokens of one review)")
plt.colorbar(label="Attention weight")
plt.tight_layout()
plt.show()

print(f"\nActual label: {'Positive' if y_test[sample_idx] == 1 else 'Negative'}")
pred_prob = transformer_model.predict(sample_review, verbose=0)[0, 0]
print(f"Model prediction: {'Positive' if pred_prob > 0.5 else 'Negative'} (confidence: {pred_prob:.3f})")


**What to look for:** Look for columns (key positions) that receive noticeably higher attention overall -- these are words the model has learned are generally informative for sentiment (e.g. strong adjectives), regardless of which word is doing the "attending".


---
## Part 8 — In-Lab Exercise (to be completed and shown to the instructor)

1. Change `num_heads` from 2 to **4** and retrain the Transformer. Does test accuracy change? Does the model summary show more or fewer total parameters, and why?
2. Stack **two** TransformerEncoderBlocks in sequence (instead of one) and retrain. Does test accuracy improve? What is the trade-off of stacking more encoder blocks?
3. Replace `GlobalAveragePooling1D()` with `GlobalMaxPooling1D()` and retrain. Compare the resulting test accuracy -- which pooling strategy works better for this task, and can you propose why?
4. Using the code from Part 7, visualize the attention pattern for a review your model classifies **incorrectly**. Does the attention pattern look noticeably different (e.g. more spread out, or focused on less obviously relevant words) compared to a correctly classified example?
5. In 4–5 sentences, summarize how the Transformer's test accuracy compared to the RNN-based models from Experiment 7, and describe -- in your own words -- one concrete advantage and one concrete disadvantage of the Transformer architecture relative to an LSTM for this specific task.


In [ ]:
# TODO 1: num_heads = 4 experiment
# your code here


_TODO 1 (continued): Your explanation here._

In [ ]:
# TODO 2: Stack two TransformerEncoderBlocks
# your code here


_TODO 2 (continued): Your explanation here._

In [ ]:
# TODO 3: GlobalMaxPooling1D experiment
# your code here


_TODO 3 (continued): Your explanation here._

In [ ]:
# TODO 4: Attention visualization for an incorrectly classified review
# your code here


_TODO 4 (continued): Your explanation here._

_TODO 5: Your summary here._

---
## Part 9 — Beyond This Lab: Pretrained Transformers

The Transformer built in this experiment was trained **entirely from scratch** on a relatively small dataset. In practice, most real-world NLP applications today instead **fine-tune a large pretrained Transformer** (such as BERT, RoBERTa, or GPT-family models) that has already learned rich language representations from massive text corpora, and only adapt it to the specific task with a much smaller amount of task-specific data. Libraries such as Hugging Face `transformers` provide easy access to hundreds of such pretrained models. This is a natural next step beyond this course, and is optional, exploratory reading rather than a graded part of this experiment.


---
## Post-Lab Viva Questions

1. What are Query, Key, and Value in self-attention, and what role does each play in computing the output?
2. Why is positional encoding necessary in a Transformer but not in an RNN?
3. What does "multi-head" attention mean, and why might using multiple heads be better than a single attention computation?
4. What is the purpose of the residual ("Add") connections and Layer Normalization in a Transformer encoder block?
5. Why can Transformer training be parallelized across the sequence dimension, while RNN training cannot?
6. Based on your results, under what circumstances might an LSTM still be a reasonable choice over a Transformer?

---
*End of Experiment 8*
